# CFPB Fraud Intelligence — End-to-End Colab Demo

This notebook clones your GitHub repository, installs the minimal runtime dependencies, and
runs the required single-command pipeline:

```bash
python src/model_runner.py
```

Before running, set the repository URL and branch in the next cell. For a private repository,
create a Colab secret named `GITHUB_TOKEN` with read access to that repository.


In [ ]:
REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPOSITORY.git"
BRANCH = "main"
PROJECT_FOLDER = "cfpb-fraud-intelligence"


In [ ]:
from pathlib import Path
import os
import subprocess

project_dir = Path("/content") / PROJECT_FOLDER

# Optional private-repository authentication through Colab Secrets.
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None

env = os.environ.copy()
if token:
    env["GITHUB_TOKEN"] = token
    askpass = Path("/tmp/git-askpass.sh")
    askpass.write_text(
        '#!/bin/sh\ncase "$1" in\n*Username*) echo "x-access-token" ;;\n*Password*) echo "$GITHUB_TOKEN" ;;\nesac\n'
    )
    askpass.chmod(0o700)
    env["GIT_ASKPASS"] = str(askpass)
    env["GIT_TERMINAL_PROMPT"] = "0"

if project_dir.exists():
    subprocess.run(["git", "-C", str(project_dir), "fetch", "origin"], check=True, env=env)
    subprocess.run(["git", "-C", str(project_dir), "checkout", BRANCH], check=True, env=env)
    subprocess.run(["git", "-C", str(project_dir), "pull", "origin", BRANCH], check=True, env=env)
else:
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, REPO_URL, str(project_dir)],
        check=True,
        env=env,
    )

os.chdir(project_dir)
print("Working directory:", Path.cwd())


In [ ]:
# Use the latest Colab runtime and install project-specific packages.
%pip install -q -r requirements.txt


In [ ]:
import torch
from pathlib import Path

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

data_path = Path("data/processed/complaints_clean.csv.gz")
print("Preprocessed dataset exists:", data_path.exists())
if data_path.exists():
    print("Dataset size (MB):", round(data_path.stat().st_size / 1_000_000, 2))


## Optional fast validation

This checks loading, sample selection, prompt construction, and output paths without
downloading FLAN-T5.


In [ ]:
!python src/model_runner.py --prepare-only

## Required full run

Use a GPU runtime (`Runtime` → `Change runtime type` → `T4 GPU`) and run the command below.
The first run downloads and caches `google/flan-t5-small`.


In [ ]:
!python src/model_runner.py

In [ ]:
from pathlib import Path
from IPython.display import Markdown, display

samples_path = Path("outputs/samples.txt")
metadata_path = Path("outputs/run_metadata.json")

print(samples_path.read_text(encoding="utf-8"))
display(Markdown(Path("outputs/output_description.md").read_text(encoding="utf-8")))
print(metadata_path.read_text(encoding="utf-8"))


## Before closing Colab

Download or commit the following small files:

- `outputs/samples.txt`
- `outputs/samples.jsonl`
- `outputs/run_metadata.json`
- `outputs/output_description.md`

Do not commit model caches or checkpoints.
